# MoE-LLaVA StableLM Colab Sanity Check

This notebook reproduces the successful Colab sanity check for `LanguageBind/MoE-LLaVA-StableLM-1.6B-4e` on a GPU runtime.

Known-good result from 2026-05-08:

- GPU: Tesla T4
- Repo: `/content/DynMoE`
- Python environment: `/content/DynMoE/.venv310`
- Hugging Face cache: `/content/hf_cache`
- Sample image: `/content/DynMoE/MoE-LLaVA/assets/image.jpg`

Detailed notes are in `/content/COLAB_SESSION_NOTES.md`. Updated handoff instructions are in `/content/COLAB_HANDOFF.md`.

Use **Runtime > Change runtime type > GPU** before running the notebook.

## 1. Runtime Checks

In [ ]:
!nvidia-smi
!python --version
!/usr/bin/python3.10 --version

## 2. Repo Setup

If `/content/DynMoE` already exists, this cell leaves it in place. Otherwise it clones the repo.

In [ ]:
%cd /content
!test -d /content/DynMoE/.git || git clone https://github.com/LINs-lab/DynMoE.git
%cd /content/DynMoE
!git status --short

## 3. Python 3.10 Venv Setup

The pinned `torch==2.0.1` stack should use Python 3.10, not Colab's default Python 3.12 environment.

In [ ]:
!apt-get update
!apt-get install -y python3.10-venv python3.10-dev
%cd /content/DynMoE
!test -x /content/DynMoE/.venv310/bin/python || python3.10 -m venv .venv310
!.venv310/bin/python --version
!.venv310/bin/python -m pip --version

## 4. Dependency Install

`--log /tmp/pip.log` avoids restricted-environment noise from pip trying to write `/var/log/pip.log`. `flash-attn` is intentionally not installed for this sanity check.

In [ ]:
%cd /content/DynMoE
!.venv310/bin/python -m pip --log /tmp/pip.log install --upgrade pip
!.venv310/bin/python -m pip --log /tmp/pip.log install torch==2.0.1 torchvision==0.15.2
!.venv310/bin/python -m pip --log /tmp/pip.log install transformers==4.37.0 tokenizers==0.15.1 sentencepiece==0.1.99
!.venv310/bin/python -m pip --log /tmp/pip.log install accelerate==0.21.0 peft==0.4.0 openpyxl==3.1.2
!.venv310/bin/python -m pip --log /tmp/pip.log install -e ./MoE-LLaVA
!.venv310/bin/python -m pip --log /tmp/pip.log install -e ./DeepSpeed-0.9.5

## 5. Verification

Expected successful checks include `torch.cuda.is_available(): True`, `deepspeed: 0.9.5+4927232`, `moellava ok`, and `builder ok`.

In [ ]:
%cd /content/DynMoE
!.venv310/bin/python -c "import torch; print('torch:', torch.__version__); print('torch.cuda.is_available():', torch.cuda.is_available())"
!.venv310/bin/python -c "import deepspeed; print('deepspeed:', deepspeed.__version__)"
!.venv310/bin/python -c "import moellava; print('moellava ok')"
!.venv310/bin/python -c "import moellava.model.builder; print('builder ok')"
!nvidia-smi

## 6. Sample Inference

Run the next cell, wait for the `USER:` prompt, then enter:

```text
Describe this image.
```

The successful previous response described a large red heart painted on a blue door. The CLI does not handle `exit` as a command after inference; use Colab's stop button or `Ctrl-C` to terminate it.

In [ ]:
%cd /content/DynMoE/MoE-LLaVA
!HF_HOME=/content/hf_cache TRANSFORMERS_CACHE=/content/hf_cache \
  /content/DynMoE/.venv310/bin/deepspeed --include localhost:0 moellava/serve/cli.py \
  --model-path "LanguageBind/MoE-LLaVA-StableLM-1.6B-4e" \
  --image-file "assets/image.jpg"

## Known Non-Blocking Warnings

The successful run showed these warnings, and they did not block inference:

- `TRANSFORMERS_CACHE` deprecation warning; `HF_HOME` is set for the durable cache path.
- `NCCL backend in DeepSpeed not yet implemented`, followed by TorchBackend NCCL initialization.
- `TypedStorage is deprecated`.
- generation warning about missing `attention_mask` and `pad_token_id`.
- `bitsandbytes` CPU/GPU support warnings during imports.